# FBref Datenakquise — Swiss Super League 2024/25

Extrahiert Team-Statistiken aus der **lokal gespeicherten FBref-HTML-Seite**.

**Voraussetzung:**  
HTML-Datei liegt in `data_acquisition/`:
```
Swiss Super League Stats _ FBref.com.html
```
Download-Datum: 21. April 2026 (nach Spieltag 33 von 36).

**Output-Dateien in `data_acquisition/raw/`:**
| Datei | Inhalt |
|---|---|
| `standard_fbref.csv` | Tore, Assists, Ballbesitz, Karten, Spielzeit |
| `keeper_fbref.csv` | Gegentore, Saves, Clean Sheets, Penalty-Saves |
| `shooting_fbref.csv` | Schüsse, Schüsse aufs Tor, xG, Umwandlungsrate |
| `misc_fbref.csv` | Fouls, Abseits, Flanken, Tackles, Eigentore |

## 1. Setup

In [ ]:
import pandas as pd
from pathlib import Path

HTML_FILE = Path("Swiss Super League Stats _ FBref.com.html")
RAW_DIR   = Path("raw")
RAW_DIR.mkdir(exist_ok=True)

assert HTML_FILE.exists(), (
    f"HTML-Datei nicht gefunden: {HTML_FILE.resolve()}\n"
    "Datei 'Swiss Super League Stats _ FBref.com.html' in data_acquisition/ ablegen."
)
print(f"✓ HTML-Datei : {HTML_FILE.resolve()}")
print(f"✓ Output-Dir : {RAW_DIR.resolve()}")

## 2. Hilfsfunktion & Tabellenkonfiguration

FBref-Tabellen haben **zweizeilige Header** (Gruppe + Spaltenname) → MultiIndex.  
Die Funktion flacht diese zu `Gruppe_Spalte` ab (z.B. `Performance_Gls`).

In [ ]:
def read_fbref_table(html_path: Path, table_id: str) -> pd.DataFrame:
    """Liest eine FBref-Tabelle via pandas.read_html() und bereinigt die Spalten.

    FBref-Besonderheiten:
    - 2-zeilige Header (Gruppe + Stat) → MultiIndex → wird zu 'Gruppe_Stat' geflacht
    - Wiederholte Header-Zeilen im tbody (Squad == 'Squad') werden entfernt
    - Totals-Zeile am Ende (Squad == '') wird entfernt
    """
    tables = pd.read_html(html_path, attrs={"id": table_id})
    if not tables:
        raise ValueError(f"Tabelle '{table_id}' nicht in der HTML-Datei gefunden.")
    df = tables[0]

    # MultiIndex-Spalten zu flachen Strings
    if isinstance(df.columns, pd.MultiIndex):
        new_cols = []
        for lvl0, lvl1 in df.columns:
            grp = "" if ("Unnamed" in str(lvl0) or str(lvl0) == str(lvl1)) else str(lvl0)
            new_cols.append(f"{grp}_{lvl1}" if grp else str(lvl1))
        df.columns = new_cols

    # Wiederholte Header-Zeilen & leere Squad-Einträge entfernen
    df = df[df["Squad"].notna() & (df["Squad"] != "Squad")].reset_index(drop=True)
    return df


# Tabellen-ID → Ausgabe-CSV
TABLES = {
    "stats_squads_standard_for": "standard_fbref.csv",
    "stats_squads_keeper_for":   "keeper_fbref.csv",
    "stats_squads_shooting_for": "shooting_fbref.csv",
    "stats_squads_misc_for":     "misc_fbref.csv",
}
print(f"Konfigurierte Tabellen ({len(TABLES)}):")
for tid, fname in TABLES.items():
    print(f"  {tid:<40} → raw/{fname}")

## 3. Tabellen extrahieren & als CSV speichern

In [ ]:
for table_id, csv_name in TABLES.items():
    print(f"\n📊 {table_id}")
    try:
        df = read_fbref_table(HTML_FILE, table_id)
        out_path = RAW_DIR / csv_name
        df.to_csv(out_path, index=False)
        print(f"   ✅ {csv_name}")
        print(f"      {len(df)} Teams × {len(df.columns)} Spalten")
        print(f"      Spalten: {', '.join(df.columns.tolist())}")
    except Exception as e:
        print(f"   ❌ Fehler: {e}")

## 4. Übersicht gespeicherter Dateien

In [ ]:
print("FBref-Dateien in data_acquisition/raw/:\n")
for f in sorted(RAW_DIR.glob("*_fbref.csv")):
    df   = pd.read_csv(f)
    size = f.stat().st_size / 1024
    print(f"  {f.name:<30} {len(df):>2} Teams × {len(df.columns):>2} Spalten  ({size:.1f} KB)")

print("\n🏁 FBref-Extraktion abgeschlossen.")
print(f"   Quelle : {HTML_FILE.name}")
print( "   Stand  : 21. April 2026 (Spieltag 33 / 36)")